In [57]:

## 2) Reference implementation (NumPy only)
import numpy as np
import math
from typing import Tuple, List

# ----------------------------
# Utilities: perfect matchings
# ----------------------------

def random_matching(num_points: int, rng: np.random.Generator) -> np.ndarray:
    """Sample a uniform random perfect matching on {0,...,num_points-1}.
    Returns an array m with m[i]=j, m[j]=i, i!=j.
    """
    assert num_points % 2 == 0
    perm = rng.permutation(num_points)
    m = np.empty(num_points, dtype=np.int32)
    for a, b in perm.reshape(-1, 2):
        m[a] = b
        m[b] = a
    return m

# ----------------------------
# Model statistics
# ----------------------------

def chords_from_matching(m: np.ndarray) -> List[Tuple[int, int]]:
    """Return list of chords (i,j) with i<j from matching array m."""
    chords = []
    seen = np.zeros(len(m), dtype=bool)
    for i in range(len(m)):
        if not seen[i]:
            j = int(m[i])
            seen[i] = True
            seen[j] = True
            if i < j:
                chords.append((i, j))
            else:
                chords.append((j, i))
    return chords

def crossing_count(m: np.ndarray) -> int:
    """Count chord crossings among chords of matching m.
    Complexity: O(N^2) with N=number of chords.
    """
    chords = chords_from_matching(m)
    X = 0
    for idx in range(len(chords)):
        i, j = chords[idx]
        for kdx in range(idx + 1, len(chords)):
            k, l = chords[kdx]
            # crossing iff i<k<j<l or k<i<l<j
            if (i < k < j < l) or (k < i < l < j):
                X += 1
    return X

def cycles_and_small_loops(m: np.ndarray, r: np.ndarray, L_star: int) -> Tuple[int, int]:
    """Return (c, N_le) where c is number of alternating cycles in union of m and r,
    and N_le counts cycles whose number of m-edges <= L_star.
    Traverses cycles by alternating m-edge then r-edge.
    Complexity: O(num_points).
    """
    n = len(m)
    visited = np.zeros(n, dtype=bool)
    c = 0
    n_small = 0
    for start in range(n):
        if visited[start]:
            continue
        # trace one cycle
        cur = start
        length_m_edges = 0
        while True:
            visited[cur] = True
            nxt = int(m[cur])  # m-edge
            length_m_edges += 1
            visited[nxt] = True
            cur = int(r[nxt])  # r-edge
            if cur == start:
                break
        c += 1
        if length_m_edges <= L_star:
            n_small += 1
    return c, n_small

# ----------------------------
# Proposals (2-switch) and MH step
# ----------------------------

def two_switch_proposal(m: np.ndarray, rng: np.random.Generator) -> Tuple[np.ndarray, Tuple[int,int,int,int]]:
    """Pick two disjoint chords (a,b) and (c,d) and rewire to either (a,c),(b,d) or (a,d),(b,c).
    Returns new matching array and the chosen quadruple (a,b,c,d) with a<b, c<d.
    """
    n = len(m)
    # Build chord list once
    chords = chords_from_matching(m)
    N = len(chords)
    i, j = rng.integers(0, N, size=2, endpoint=False)
    while j == i:
        j = rng.integers(0, N, endpoint=False)
    (a, b) = chords[i]
    (c, d) = chords[j]
    # rewire option
    if rng.random() < 0.5:
        # (a,c),(b,d)
        new_pairs = [(min(a, c), max(a, c)), (min(b, d), max(b, d))]
        connect = [(a, c), (b, d)]
    else:
        # (a,d),(b,c)
        new_pairs = [(min(a, d), max(a, d)), (min(b, c), max(b, c))]
        connect = [(a, d), (b, c)]
    m2 = m.copy()
    # disconnect old
    m2[a] = a  # temp mark; will be overwritten
    m2[b] = b
    m2[c] = c
    m2[d] = d
    # connect new
    for u, v in connect:
        m2[u] = v
        m2[v] = u
    return m2, (a, b, c, d)


def mh_step(m: np.ndarray, r: np.ndarray, L_star: int, lam: float, alpha: float,
            rng: np.random.Generator,
            cached_stats: Tuple[int,int,int] = None) -> Tuple[np.ndarray, Tuple[int,int,int]]:
    """One Metropolis–Hastings step at fixed (lam, alpha).
    cached_stats = (X, c, n_small) for current m may be provided to save recomputation.
    Returns (m_next, stats_next).
    """
    if cached_stats is None:
        X = crossing_count(m)
        c, n_small = cycles_and_small_loops(m, r, L_star)
    else:
        X, c, n_small = cached_stats

    m_prop, _ = two_switch_proposal(m, rng)
    Xp = crossing_count(m_prop)
    cp, nsp = cycles_and_small_loops(m_prop, r, L_star)

    dX = Xp - X
    dc = cp - c
    dn = nsp - n_small

    log_ratio = -(lam * dX + alpha * dn) - (dc * math.log(2.0))
    if log_ratio >= 0 or math.log(rng.random()) < log_ratio:
        return m_prop, (Xp, cp, nsp)
    else:
        return m, (X, c, n_small)

# ----------------------------
# Thermodynamic integration
# ----------------------------

def base_logZ_N(N: int) -> float:
    """log Z_N(0,0) = 3 N log 2 + lgamma(N+1/4) - lgamma(1/4)."""
    return 3 * N * math.log(2.0) + math.lgamma(N + 0.25) - math.lgamma(0.25)


def mcmc_expectations(N: int, lam: float, alpha: float, L_star: int, sweeps: int,
                      rng: np.random.Generator,
                      m_init: np.ndarray = None,
                      r: np.ndarray = None,
                      burn_sweeps: int = None) -> Tuple[float, float, float, np.ndarray, np.ndarray]:
    """Run MCMC at fixed (lam, alpha). Returns (E[X], E[N_le], E[c], m_last, r).
    One sweep = 2N proposals. If m_init is None, initialize randomly. If r is None, sample rods once.
    """
    if r is None:
        r = random_matching(2 * N, rng)
    if m_init is None:
        m = random_matching(2 * N, rng)
    else:
        m = m_init.copy()

    if burn_sweeps is None:
        burn_sweeps = max(5, sweeps // 10)

    X, c, n_small = crossing_count(m), *cycles_and_small_loops(m, r, L_star)

    def do_sweep(m, stats):
        X, c, n_small = stats
        for _ in range(2 * N):
            m, (X, c, n_small) = mh_step(m, r, L_star, lam, alpha, rng, (X, c, n_small))
        return m, (X, c, n_small)

    # burn-in
    for _ in range(burn_sweeps):
        m, (X, c, n_small) = do_sweep(m, (X, c, n_small))

    # collect
    sum_X = 0.0
    sum_n = 0.0
    sum_c = 0.0
    samples = 0
    for _ in range(sweeps):
        m, (X, c, n_small) = do_sweep(m, (X, c, n_small))
        sum_X += X
        sum_n += n_small
        sum_c += c
        samples += 1
    return (sum_X / samples, sum_n / samples, sum_c / samples, m, r)


def trapezoid(xs: np.ndarray, ys: np.ndarray) -> float:
    return float(np.trapz(ys, xs))


def thermo_integrate(N: int, L_star: int, lam_grid: np.ndarray, alpha_grid: np.ndarray,
                     sweeps: int, rng: np.random.Generator) -> Tuple[float, dict]:
    """Follow path (0,0)->(lam_max,0)->(lam_max,alpha_max) and return
    logZ(lam_max, alpha_max) and a dict of diagnostics with grids and expectations.
    """
    # Fixed rod structure sampled once
    r = random_matching(2 * N, rng)

    # Stage 1: alpha=0, vary lambda
    EX = []
    m = None
    for lam in lam_grid:
        ex, en, ec, m, _ = mcmc_expectations(N, lam, 0.0, L_star, sweeps, rng, m_init=m, r=r)
        EX.append(ex)
    EX = np.array(EX)
    I_lambda = trapezoid(lam_grid, EX)  # integral of E[X] d lambda

    # Stage 2: lambda = lam_max, vary alpha
    lam_max = float(lam_grid[-1])
    EN = []
    for alpha in alpha_grid:
        ex, en, ec, m, _ = mcmc_expectations(N, lam_max, alpha, L_star, sweeps, rng, m_init=m, r=r)
        EN.append(en)
    EN = np.array(EN)
    I_alpha = trapezoid(alpha_grid, EN)  # integral of E[N_le] d alpha

    logZ00 = base_logZ_N(N)
    logZ = logZ00 - I_lambda - I_alpha

    diag = {
        "lam_grid": lam_grid, "EX": EX,
        "alpha_grid": alpha_grid, "EN": EN,
        "I_lambda": I_lambda, "I_alpha": I_alpha,
        "logZ00": logZ00,
    }
    return logZ, diag

# ----------------------------
# Example usage (lightweight demo)
# ----------------------------
if __name__ == "__main__":
    rng = np.random.default_rng(1924)
    N = 25          # try 200 for the real run
    L_star = 12
    lam_grid = np.linspace(0.0, 1.0, 10)     # refine as needed
    alpha_grid = np.linspace(0.0, 0.5, 10)   # refine as needed
    sweeps = 100                              # increase (e.g., 2000+) for production

    logZ, diag = thermo_integrate(N, L_star, lam_grid, alpha_grid, sweeps, rng)
    print(f"log Z_N(lam={lam_grid[-1]:.3f}, alpha={alpha_grid[-1]:.3f}) ≈ {logZ:.4f}")
    print("I_lambda=", diag["I_lambda"], "I_alpha=", diag["I_alpha"], "logZ00=", diag["logZ00"])

log Z_N(lam=1.000, alpha=0.500) ≈ 71.1509
I_lambda= 34.812777777777775 I_alpha= 0.32 logZ00= 106.28370206216725


In [ ]:
68.9157
69.2815
70
10 grid
71.6552
71.1721
71.3833
71.3248

In [62]:
import numpy as np
from scipy.special import gammaln
import math

def generate_random_matching(n_points):
    """Generate a random perfect matching of n_points."""
    points = list(range(n_points))
    np.random.shuffle(points)
    matching = {}
    for i in range(0, n_points, 2):
        matching[points[i]] = points[i+1]
        matching[points[i+1]] = points[i]
    return matching

def count_crossings(pairing, n_points):
    """Count the number of crossing chords in a pairing."""
    chords = []
    seen = set()
    for i in range(n_points):
        if i not in seen:
            j = pairing[i]
            if i < j:
                chords.append((i, j))
            else:
                chords.append((j, i))
            seen.add(i)
            seen.add(j)
    
    crossings = 0
    for idx1, (i, j) in enumerate(chords):
        for idx2 in range(idx1 + 1, len(chords)):
            k, l = chords[idx2]
            # Check if chords cross
            if (i < k < j < l) or (k < i < l < j):
                crossings += 1
    return crossings

def count_loops_and_lengths(rod, pairing, n_points):
    """Count loops and their lengths in the union of rod and pairing."""
    visited = set()
    loops = []
    
    for start in range(n_points):
        if start in visited:
            continue
        
        loop = []
        current = start
        use_rod = True
        
        while True:
            loop.append(current)
            visited.add(current)
            
            if use_rod:
                current = rod[current]
            else:
                current = pairing[current]
            
            use_rod = not use_rod
            
            if current == start:
                break
        
        # Length in terms of M-edges (pairing edges)
        loop_length_M = len(loop) // 2
        loops.append(loop_length_M)
    
    return len(loops), loops

def count_small_loops(rod, pairing, n_points, L_star):
    """Count loops with length <= L_star."""
    _, loop_lengths = count_loops_and_lengths(rod, pairing, n_points)
    return sum(1 for length in loop_lengths if length <= L_star)

def metropolis_swap(pairing, rod, n_points, lambda_val, alpha_val, L_star):
    """Perform a Metropolis-Hastings swap move."""
    # Choose two random chords to swap
    points = list(range(n_points))
    np.random.shuffle(points)
    a, b, c, d = points[:4]
    
    # Check if (a,b) and (c,d) are both chords
    if pairing[a] != b or pairing[c] != d:
        # Try to find valid chords
        a = np.random.randint(n_points)
        b = pairing[a]
        remaining = [i for i in range(n_points) if i != a and i != b]
        if len(remaining) < 2:
            return pairing
        c = np.random.choice(remaining)
        d = pairing[c]
    
    # Create new pairing with swapped connections
    new_pairing = pairing.copy()
    new_pairing[a] = c
    new_pairing[c] = a
    new_pairing[b] = d
    new_pairing[d] = b
    
    # Calculate energy difference
    old_crossings = count_crossings(pairing, n_points)
    new_crossings = count_crossings(new_pairing, n_points)
    
    old_c, _ = count_loops_and_lengths(rod, pairing, n_points)
    new_c, _ = count_loops_and_lengths(rod, new_pairing, n_points)
    
    old_small = count_small_loops(rod, pairing, n_points, L_star)
    new_small = count_small_loops(rod, new_pairing, n_points, L_star)
    
    # Energy difference
    delta_E = lambda_val * (new_crossings - old_crossings) + \
              alpha_val * (new_small - old_small)
    
    # Weight difference (from the 2^(2N-c(M)) factor)
    delta_weight = (old_c - new_c) * np.log(2)
    
    # Metropolis acceptance
    if np.random.random() < np.exp(-delta_E + delta_weight):
        return new_pairing
    return pairing

def run_mcmc(rod, n_points, lambda_val, alpha_val, L_star, n_steps=10000, burn_in=2000):
    """Run MCMC to sample pairings and estimate expectations."""
    pairing = generate_random_matching(n_points)
    
    samples_X = []
    samples_N = []
    
    for step in range(n_steps):
        pairing = metropolis_swap(pairing, rod, n_points, lambda_val, alpha_val, L_star)
        
        if step >= burn_in:
            X = count_crossings(pairing, n_points)
            N = count_small_loops(rod, pairing, n_points, L_star)
            samples_X.append(X)
            samples_N.append(N)
    
    return np.mean(samples_X), np.mean(samples_N)

def compute_log_Z(N, lambda_final, alpha_final, L_star, lam_grid, alpha_grid, seed):
    """Compute log partition function using thermodynamic integration."""
    np.random.seed(seed)
    
    n_points = 2 * N
    
    # Generate random rod structure
    rod = generate_random_matching(n_points)
    
    # Base partition function at (0,0)
    log_Z_00 = 3 * N * np.log(2) + gammaln(N + 0.25) - gammaln(0.25)
    
    # First integrate lambda from 0 to lambda_final with alpha=0
    E_X_values = []
    for lam in lam_grid:
        E_X, _ = run_mcmc(rod, n_points, lam, 0.0, L_star)
        E_X_values.append(E_X)
    
    # Trapezoidal integration
    integral_lambda = np.trapz(E_X_values, lam_grid)
    
    # Then integrate alpha from 0 to alpha_final with lambda=lambda_final
    E_N_values = []
    for alpha in alpha_grid:
        _, E_N = run_mcmc(rod, n_points, lambda_final, alpha, L_star)
        E_N_values.append(E_N)
    
    # Trapezoidal integration
    integral_alpha = np.trapz(E_N_values, alpha_grid)
    
    # Final result
    log_Z = log_Z_00 - integral_lambda - integral_alpha
    
    return log_Z

# Main computation
N = 25
L_star = 12
lam_grid = np.linspace(0.0, 1.0, 10)
alpha_grid = np.linspace(0.0, 0.5, 10)
lambda_final = 1.0
alpha_final = 0.5

# Average over 3 random seeds
results = []
for seed in [42, 123, 456]:
    log_Z = compute_log_Z(N, lambda_final, alpha_final, L_star, lam_grid, alpha_grid, seed)
    results.append(log_Z)
    print(f"Seed {seed}: log Z = {log_Z:.2f}")

final_result = np.mean(results)
print(f"\nFinal result: log Z_N(1.0, 0.5) = {final_result:.0f}")

Seed 42: log Z = 71.68
Seed 123: log Z = 71.74
Seed 456: log Z = 71.95

Final result: log Z_N(1.0, 0.5) = 72


In [70]:
import numpy as np
from scipy.special import loggamma
import math

# Set parameters
N = 25
L_star = 12
lam_grid = np.linspace(0.0, 1.0, 10)
alpha_grid = np.linspace(0.0, 0.5, 10)

# Generate random rod structure
def generate_random_rod():
    """Generate a random perfect matching (rod structure)"""
    points = list(range(2*N))
    np.random.shuffle(points)
    rod = [-1] * (2*N)
    for i in range(0, 2*N, 2):
        rod[points[i]] = points[i+1]
        rod[points[i+1]] = points[i]
    return rod

# Count crossings between two chords
def chords_cross(i, j, k, l):
    """Check if chords (i,j) and (k,l) cross"""
    if i > j: i, j = j, i
    if k > l: k, l = l, k
    return (i < k < j < l) or (k < i < l < j)

# Count total crossings in a pairing
def count_crossings(pairing):
    """Count crossing pairs in pairing"""
    chords = []
    seen = set()
    for i in range(2*N):
        if i not in seen:
            j = pairing[i]
            seen.add(i)
            seen.add(j)
            chords.append((min(i,j), max(i,j)))
    
    crossings = 0
    for idx1 in range(len(chords)):
        for idx2 in range(idx1+1, len(chords)):
            i, j = chords[idx1]
            k, l = chords[idx2]
            if chords_cross(i, j, k, l):
                crossings += 1
    return crossings

# Find all loops and their properties
def analyze_loops(rod, pairing):
    """Find loops and compute c(M) and N_≤L*(M)"""
    visited = [False] * (2*N)
    num_loops = 0
    short_loops = 0
    
    for start in range(2*N):
        if visited[start]:
            continue
        
        # Trace the loop
        loop_length = 0
        current = start
        use_rod = True
        
        while True:
            visited[current] = True
            
            if use_rod:
                current = rod[current]
            else:
                current = pairing[current]
                loop_length += 1  # Count M-edges
            
            use_rod = not use_rod
            
            if current == start:
                break
        
        num_loops += 1
        if loop_length <= L_star:
            short_loops += 1
    
    return num_loops, short_loops

# MCMC swap move
def mcmc_swap(pairing, rod, lam, alpha, beta=1.0):
    """Perform MCMC swap of two random pairs"""
    # Pick two edges to swap
    edges = []
    seen = set()
    for i in range(2*N):
        if i not in seen:
            j = pairing[i]
            edges.append((i, j))
            seen.add(i)
            seen.add(j)
    
    if len(edges) < 2:
        return pairing
    
    # Select two random edges
    idx1, idx2 = np.random.choice(len(edges), 2, replace=False)
    a, b = edges[idx1]
    c, d = edges[idx2]
    
    # Create new pairing with swapped connections
    new_pairing = pairing.copy()
    new_pairing[a] = c
    new_pairing[c] = a
    new_pairing[b] = d
    new_pairing[d] = b
    
    # Compute energy change
    old_X = count_crossings(pairing)
    new_X = count_crossings(new_pairing)
    
    old_c, old_N = analyze_loops(rod, pairing)
    new_c, new_N = analyze_loops(rod, new_pairing)
    
    # Energy difference
    delta_E = lam * (new_X - old_X) + alpha * (new_N - old_N)
    
    # Weight factor difference (from parity constraint)
    delta_log_weight = (old_c - new_c) * np.log(2)
    
    # Metropolis acceptance
    if np.random.random() < np.exp(-beta * delta_E + delta_log_weight):
        return new_pairing
    return pairing

# Run MCMC sampling
def estimate_expectations(rod, lam, alpha, n_steps=20000, n_burn=5000):
    """Estimate expectations using MCMC"""
    # Initialize with random pairing
    points = list(range(2*N))
    np.random.shuffle(points)
    pairing = [-1] * (2*N)
    for i in range(0, 2*N, 2):
        pairing[points[i]] = points[i+1]
        pairing[points[i+1]] = points[i]
    
    # Burn-in
    for _ in range(n_burn):
        pairing = mcmc_swap(pairing, rod, lam, alpha)
    
    # Collect samples
    X_sum = 0
    N_sum = 0
    
    for _ in range(n_steps):
        pairing = mcmc_swap(pairing, rod, lam, alpha)
        X_sum += count_crossings(pairing)
        _, N_short = analyze_loops(rod, pairing)
        N_sum += N_short
    
    return X_sum / n_steps, N_sum / n_steps

# Perform thermodynamic integration
def compute_log_Z():
    """Compute log Z_N(1.0, 0.5) via thermodynamic integration"""
    
    # Generate random rod structure
    np.random.seed(42)  # For reproducibility
    rod = generate_random_rod()
    
    # Starting point: log Z(0,0)
    log_Z = 3 * N * np.log(2) + loggamma(N + 0.25) - loggamma(0.25)
    
    # Path 1: Integrate λ from 0 to 1.0 at α=0
    print("Integrating lambda from 0 to 1.0...")
    for i in range(len(lam_grid)-1):
        lam_mid = (lam_grid[i] + lam_grid[i+1]) / 2
        mean_X, _ = estimate_expectations(rod, lam_mid, 0.0)
        d_lam = lam_grid[i+1] - lam_grid[i]
        log_Z -= mean_X * d_lam
        print(f"  λ={lam_mid:.2f}: ⟨X⟩={mean_X:.2f}")
    
    # Path 2: Integrate α from 0 to 0.5 at λ=1.0  
    print("\nIntegrating alpha from 0 to 0.5...")
    for i in range(len(alpha_grid)-1):
        alpha_mid = (alpha_grid[i] + alpha_grid[i+1]) / 2
        _, mean_N = estimate_expectations(rod, 1.0, alpha_mid)
        d_alpha = alpha_grid[i+1] - alpha_grid[i]
        log_Z -= mean_N * d_alpha
        print(f"  α={alpha_mid:.3f}: ⟨N_≤L*⟩={mean_N:.2f}")
    
    return log_Z

# Main computation
result = compute_log_Z()
print(f"\nFinal result: log Z_N(λ=1.0, α=0.5) = {result:.0f}")

Integrating lambda from 0 to 1.0...
  λ=0.06: ⟨X⟩=74.09
  λ=0.17: ⟨X⟩=51.18
  λ=0.28: ⟨X⟩=38.56
  λ=0.39: ⟨X⟩=29.81
  λ=0.50: ⟨X⟩=23.51
  λ=0.61: ⟨X⟩=19.92
  λ=0.72: ⟨X⟩=17.73
  λ=0.83: ⟨X⟩=14.71
  λ=0.94: ⟨X⟩=12.87

Integrating alpha from 0 to 0.5...
  α=0.028: ⟨N_≤L*⟩=0.77
  α=0.083: ⟨N_≤L*⟩=0.78
  α=0.139: ⟨N_≤L*⟩=0.81
  α=0.194: ⟨N_≤L*⟩=0.68
  α=0.250: ⟨N_≤L*⟩=0.61
  α=0.306: ⟨N_≤L*⟩=0.54
  α=0.361: ⟨N_≤L*⟩=0.54
  α=0.417: ⟨N_≤L*⟩=0.50
  α=0.472: ⟨N_≤L*⟩=0.52

Final result: log Z_N(λ=1.0, α=0.5) = 75


In [15]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import math
import random
from collections import defaultdict

# ============================================================
# Geometry primitives
# ============================================================

EPS = 1e-10
TAU = 2.0 * math.pi

def dot(a, b):
    return a[0]*b[0] + a[1]*b[1]

def cross(a, b):
    return a[0]*b[1] - a[1]*b[0]

def sub(a, b):
    return (a[0]-b[0], a[1]-b[1])

def seg_intersection(p, r, q, s):
    """
    Solve p + t r = q + u s for t,u in (0,1).
    Returns (ok, point, t, u) with strict interior test (excluding endpoints).
    """
    rxs = cross(r, s)
    qp = sub(q, p)
    if abs(rxs) < EPS:
        return (False, None, None, None)  # parallel or collinear (measure-zero for random chords)
    t = cross(qp, s) / rxs
    u = cross(qp, r) / rxs
    if t > EPS and t < 1.0 - EPS and u > EPS and u < 1.0 - EPS:
        ip = (p[0] + t*r[0], p[1] + t*r[1])
        return (True, ip, t, u)
    return (False, None, None, None)

def polygon_area(coords, poly_ids):
    """Shoelace area for a polygon listed as vertex ids in order (no repeated last)."""
    n = len(poly_ids)
    if n < 3:
        return 0.0
    s = 0.0
    for i in range(n):
        x1, y1 = coords[poly_ids[i]]
        x2, y2 = coords[poly_ids[(i+1) % n]]
        s += x1*y2 - x2*y1
    return abs(s) * 0.5

# ============================================================
# Build faces from chords via half-edge walk
# ============================================================

def build_faces_energy_from_pairs(N, pairs, circle_points, beta=1.0):
    """
    Given N points on the unit circle and a perfect matching 'pairs' (list of (i,j)),
    1) build all chord-chord intersections,
    2) build the planar graph (vertices: endpoints+crossings; edges: chord pieces),
    3) enumerate *bounded internal polygonal* faces (no boundary-arc faces),
    4) compute energy E = sum_{faces} sign * exp(area), sign=+ if even corners else -.
    Returns (E, faces_list) where faces_list = [(corners, area), ...].
    """
    # 0) Vertex list: first N are the circle points
    coords = list(circle_points)  # id: 0..N-1 are boundary points
    next_vid = N

    # 1) Gather intersections for each chord
    # chord i: endpoints a,b with indices in [0,N-1]
    chords = []
    for (a, b) in pairs:
        pa, pb = coords[a], coords[b]
        chords.append((a, b, pa, pb))

    # For each chord, store internal division points as (t, vertex_id)
    cuts = [ [] for _ in range(len(chords)) ]
    # Map (i,j) -> intersection vertex id (i<j) to keep uniqueness
    inter_vid = {}

    for i in range(len(chords)):
        a1, b1, p1, p2 = chords[i]
        r = sub(p2, p1)
        for j in range(i+1, len(chords)):
            a2, b2, q1, q2 = chords[j]
            s = sub(q2, q1)
            ok, ip, t, u = seg_intersection(p1, r, q1, s)
            if ok:
                vid = next_vid
                next_vid += 1
                coords.append(ip)
                inter_vid[(i, j)] = vid
                cuts[i].append((t, vid))
                cuts[j].append((u, vid))

    # 2) Build edges (subsegments) along each chord
    # adjacency for undirected graph
    adj = defaultdict(set)
    directed_edges = set()

    for idx, (a, b, pa, pb) in enumerate(chords):
        # Add endpoints with parameters along the chord
        # Choose parameterization: t=0 at pa (endpoint a), t=1 at pb (endpoint b)
        pieces = [(0.0, a)] + sorted(cuts[idx], key=lambda x: x[0]) + [(1.0, b)]
        # Create edges between consecutive vertices along the chord
        for k in range(len(pieces)-1):
            u_id = pieces[k][1]
            v_id = pieces[k+1][1]
            if u_id == v_id:
                continue
            adj[u_id].add(v_id)
            adj[v_id].add(u_id)
            directed_edges.add((u_id, v_id))
            directed_edges.add((v_id, u_id))

    # 3) Precompute angular order of neighbors per vertex
    neighbor_order = {}
    for v, nbrs in adj.items():
        cx, cy = coords[v]
        angs = []
        for w in nbrs:
            wx, wy = coords[w]
            ang = math.atan2(wy - cy, wx - cx)
            angs.append((ang, w))
        # sort by angle in [-pi, pi)
        angs.sort()
        neighbor_order[v] = [w for (ang, w) in angs]

    # Helpers to find "next edge" at a vertex by left-turn rule
    def next_edge(u, v):
        """
        Given directed edge (u->v), at vertex v choose outgoing edge (v->w)
        whose direction has the smallest positive angle delta from the incoming direction.
        This walks faces with the interior on the left, yielding CCW polygons.
        """
        vx, vy = coords[v]
        ang_in = math.atan2(coords[u][1] - vy, coords[u][0] - vx)  # direction of incoming into v
        # among neighbors of v, pick w minimizing delta = (ang_out - ang_in) mod 2pi, delta>0
        best_w = None
        best_delta = None
        for w in neighbor_order[v]:
            if w == u:
                # allowed; the angular test will deal with it (but it would trace back)
                pass
            wx, wy = coords[w]
            ang_out = math.atan2(wy - vy, wx - vx)
            delta = (ang_out - ang_in) % TAU
            if delta <= EPS:  # want strictly positive turn
                delta += TAU
            if best_delta is None or delta < best_delta:
                best_delta = delta
                best_w = w
        return best_w

    # 4) Face-walk over half-edges
    visited = set()
    faces = []  # list of polygons as list of vertex ids in order

    for (u0, v0) in list(directed_edges):
        if (u0, v0) in visited:
            continue
        # Walk this half-edge to form a face
        poly = []
        u, v = u0, v0
        # Put a sane upper bound on steps to avoid infinite loops in degenerate cases
        for _ in range(10_000):
            poly.append(u)
            w = next_edge(u, v)
            if w is None:
                poly = []  # failed face
                break
            visited.add((u, v))
            u, v = v, w
            if (u, v) == (u0, v0):
                # close the polygon
                break
        else:
            poly = []

        if not poly:
            continue
        # Remove the duplicate last vertex if present
        if poly[0] == poly[-1]:
            poly = poly[:-1]
        # Basic validity
        if len(poly) < 3:
            continue

        # Keep only faces strictly internal (no boundary circle vertices)
        if any(vid < N for vid in poly):
            continue

        # Area filter to drop near-degenerate slivers
        A = polygon_area(coords, poly)
        if A <= 1e-12:
            continue

        faces.append((poly, A))

    # 5) Energy: ±exp(area) with + for even-corner faces, − for odd-corner faces
    E = 0.0
    for poly, A in faces:
        k = len(poly)
        sign = +1.0 if (k % 2 == 0) else -1.0
        E += sign * math.exp(A)

    # For completeness, return list of (corners, area) as well if needed
    return E, [(len(poly), A) for (poly, A) in faces]

# ============================================================
# Sampling & thermodynamics
# ============================================================

def double_factorial_odd(n_minus_1):
    """(N-1)!! for even N (n_minus_1 = N-1)."""
    n = n_minus_1
    if n < 1:
        return 1
    prod = 1
    while n > 0:
        prod *= n
        n -= 2
    return prod

def random_perfect_matching(N):
    """Uniform random pairing of 0..N-1."""
    idx = list(range(N))
    random.shuffle(idx)
    pairs = [(idx[2*i], idx[2*i+1]) for i in range(N // 2)]
    return pairs

def circle_points(N, radius=1.0):
    """Equally spaced points on a unit circle (counterclockwise)."""
    pts = []
    for k in range(N):
        theta = TAU * (k / N)
        pts.append((radius * math.cos(theta), radius * math.sin(theta)))
    return pts

def logsumexp(values):
    """Return log(sum_i exp(values_i)) stably."""
    if not values:
        return -math.inf
    m = max(values)
    if not math.isfinite(m):
        return m
    s = sum(math.exp(v - m) for v in values)
    return m + math.log(s)

def estimate_Z_and_Eavg(N, beta=1.0, samples=400, rng_seed=None):
    """
    Monte Carlo estimate of:
      Z(beta) ≈ (N-1)!! * mean_{uniform pairings} [exp(-beta E)]
      <E> ≈ sum E exp(-beta E) / sum exp(-beta E)
    Also returns the energy of the last sampled configuration (for reference).
    """
    if N % 2 == 1 or N < 2:
        raise ValueError("N must be an even integer ≥ 2.")

    if rng_seed is not None:
        random.seed(rng_seed)

    pts = circle_points(N)
    log_weights = []
    energies = []

    last_E = None

    for _ in range(samples):
        pairs = random_perfect_matching(N)
        E, _faces = build_faces_energy_from_pairs(N, pairs, pts, beta=beta)
        last_E = E
        energies.append(E)
        log_weights.append(-beta * E)

    # <E> using normalized log-weights
    L = logsumexp(log_weights)
    # normalize weights: w_i / Σ w
    weighted_E_num = 0.0
    for Ei, li in zip(energies, log_weights):
        wi = math.exp(li - L)
        weighted_E_num += Ei * wi
    E_avg = weighted_E_num  # already normalized

    # Z estimate
    matchings = double_factorial_odd(N - 1)  # (N-1)!!
    log_mean_w = logsumexp(log_weights) - math.log(samples)
    log_Z_est = math.log(matchings) + log_mean_w

    # Return Z (if finite) and logZ to be safe
    Z_est = math.exp(log_Z_est) if log_Z_est < 700.0 else float("inf")

    return {
        "E_last": last_E,
        "E_avg": E_avg,
        "Z_est": Z_est,
        "logZ_est": log_Z_est,
        "samples": samples,
        "beta": beta,
        "matchings": matchings,
    }

# ============================================================
# CLI
# ============================================================

def main():
    print("=== Random chords on a circle: Energy and Partition Function (Monte Carlo) ===")
    try:
        N = int(input("Enter an even number of points N (e.g., 12): ").strip())
    except Exception:
        print("Invalid input.")
        return
    if N % 2 == 1 or N < 2:
        print("Error: N must be an even integer ≥ 2.")
        return

    # Defaults per your request: only input is N
    BETA = 1.0
    # Choose a sample count that balances speed/variance; you can change this constant if desired.
    SAMPLES = 400 if N <= 40 else 200

    out = estimate_Z_and_Eavg(N, beta=BETA, samples=SAMPLES)

    print(f"\nβ = {out['beta']}")
    print(f"samples = {out['samples']}, number of perfect matchings = (N-1)!! = {out['matchings']}")
    print(f"Energy of last sampled configuration E(C_sample) = {out['E_last']:.10f}")
    print(f"Estimated average energy ⟨E⟩ ≈ {out['E_avg']:.10f}")
    if math.isfinite(out["Z_est"]):
        print(f"Estimated partition function Z(β) ≈ {out['Z_est']:.10e}")
    else:
        print("Estimated partition function is very large; reporting log Z instead.")
    print(f"log Z(β) ≈ {out['logZ_est']:.10f}")

if __name__ == "__main__":
    main()


=== Random chords on a circle: Energy and Partition Function (Monte Carlo) ===

β = 1.0
samples = 400, number of perfect matchings = (N-1)!! = 654729075
Energy of last sampled configuration E(C_sample) = -3.3572626307
Estimated average energy ⟨E⟩ ≈ -12.9444435607
Estimated partition function Z(β) ≈ 1.9586638753e+12
log Z(β) ≈ 28.3032836604


In [3]:
import numpy as np

# ---------------------------
# Model + simulation settings
# ---------------------------
L = 64                       # lattice size 
J = 0.05                     # interaction strength
alpha = 1.0                  # on-site potential scale in V(u)=alpha*(u**4 - u**2)
lam = 2.0                    # decay length in w(r)=exp(-r/lam)
R_c = 10.0                   # cutoff radius for kernel (<= lam to keep it local+fast)
temps = np.linspace(5.0, 0.01, 50)   # temperature grid (high -> low)
N_eq = 400                   # equilibration sweeps per T
N_meas = 400                 # measurement sweeps per T
seeds = [123456, 789011, 424242]      # three RNG seeds

# Discrete 7-level set and on-site potential
STATES = np.array([-3, -2, -1, 0, 1, 2, 3], dtype=np.float64)
V_STATES = alpha * (STATES**4 - STATES**2)             # shape (7,)

# ---------------------------
# Build periodic kernel w(x,y)
# ---------------------------
def build_kernel(L, lam=10.0, R_c=10.0):
    # periodic minimum-image distances
    dx = np.minimum(np.arange(L), L - np.arange(L))
    dy = dx.copy()
    X, Y = np.meshgrid(dx, dy, indexing='xy')
    R = np.hypot(X, Y).astype(np.float64)
    W = np.exp(-R / lam)
    W[R == 0] = 0.0  # no self-interaction
    if R_c is not None and R_c > 0:
        W = np.where(R <= R_c, W, 0.0)
    return W

W = build_kernel(L, lam=lam, R_c=R_c)
W_fft = np.fft.fft2(W)  # precompute FFT of kernel
N_sites = L * L

# -------------
# Observables helpers
# -------------
def energy_total(x, h):
    """
    Total energy H(x) = sum_i V(x_i) - 0.5 * sum_i x_i * (J*sum_j w_ij x_j)
                      = sum_i V(x_i) - 0.5 * sum_i x_i * h_i
    where h is the field computed with the current x (w(0)=0).
    """
    Vsum = (alpha * (x**4 - x**2)).sum()
    pair = -0.5 * np.sum(x * h)
    return Vsum + pair  # scalar (total energy)

def binder_u4_from_moments(m2, m4):
    return 1.0 - (m4 / (3.0 * (m2**2) + 1e-300))

# -------------
# Heat-bath step (parallel / synchronous)
# -------------
def heat_bath_parallel(x, beta, J, W_fft, rng):
    """
    Perform one parallel heat-bath sweep using FFT-accelerated field.
    x: (L,L) current field
    Returns: (x_new, h_new) after the sweep.
    """
    # compute local field h = J * (w * x) via circular convolution
    h = J * np.fft.ifft2(np.fft.fft2(x) * W_fft).real   # (L,L)

    # sitewise categorical sampling over the 7 levels
    # logits(u) = -beta * [ V(u) - h * u ]
    logits = -beta * (V_STATES[:, None, None] - h[None, :, :] * STATES[:, None, None])
    # numerical stability
    logits -= logits.max(axis=0, keepdims=True)
    probs = np.exp(logits)
    probs_sum = probs.sum(axis=0, keepdims=True)
    probs /= probs_sum

    # sample category per site using inverse-CDF
    cdf = np.cumsum(probs, axis=0)  # shape (7,L,L), last slice should be ~1
    U = rng.random(size=x.shape)
    # Guard against rare numerical drift
    U = np.minimum(U, 1.0 - 1e-12)
    k = np.argmax(cdf >= U[None, :, :], axis=0)
    x_new = STATES[k]

    # field for the new configuration (needed for energy and next step)
    h_new = J * np.fft.ifft2(np.fft.fft2(x_new) * W_fft).real
    return x_new, h_new

# ---------------------------
# Jackknife utilities
# ---------------------------
def jackknife_stats_m_series(m, beta):
    """
    Given m time series (length n), return:
      - m_abs_mean and its jackknife stderr
      - chi and its jackknife stderr
      - U4 and its jackknife stderr
    """
    m = np.asarray(m)
    n = m.size
    abs_m = np.abs(m)

    # Precompute moments
    S1 = m.sum()
    S2 = (m*m).sum()
    S4 = (m**4).sum()
    Sabs = abs_m.sum()

    # Point estimates
    mean_m   = S1 / n
    mean_m2  = S2 / n
    mean_m4  = S4 / n
    m_abs_mean = Sabs / n

    chi = beta * N_sites * (mean_m2 - mean_m**2)
    U4 = binder_u4_from_moments(mean_m2, mean_m4)

    # Jackknife leave-one-out means (vectorized)
    n1 = n - 1.0
    mean_m_lo   = (S1 - m)   / n1
    mean_m2_lo  = (S2 - m*m) / n1
    mean_m4_lo  = (S4 - m**4)/ n1
    mean_abs_lo = (Sabs - abs_m) / n1

    chi_lo = beta * N_sites * (mean_m2_lo - mean_m_lo**2)
    U4_lo  = 1.0 - (mean_m4_lo / (3.0 * (mean_m2_lo**2) + 1e-300))
    mabs_lo = mean_abs_lo

    # Jackknife variance: (n-1)/n * sum (theta_lo - mean(theta_lo))^2
    def jk_stderr(theta_lo):
        tbar = theta_lo.mean()
        var = (n - 1.0) / n * np.mean((theta_lo - tbar)**2)
        return np.sqrt(var)

    m_abs_se = jk_stderr(mabs_lo)
    chi_se   = jk_stderr(chi_lo)
    U4_se    = jk_stderr(U4_lo)

    return (m_abs_mean, m_abs_se), (chi, chi_se), (U4, U4_se)

def jackknife_stats_energy_series(H, beta):
    """
    Given total energy H time series (length n), return:
      - C (specific heat per spin) and jackknife stderr
      - E_per_spin mean and jackknife stderr
    Using:
      C = beta^2 / N * ( <H^2> - <H>^2 )
      e = <H>/N
    """
    H = np.asarray(H)
    n = H.size
    S1 = H.sum()
    S2 = (H*H).sum()

    mean_H  = S1 / n
    mean_H2 = S2 / n

    C = (beta**2 / N_sites) * (mean_H2 - mean_H**2)
    e_mean = mean_H / N_sites

    n1 = n - 1.0
    mean_H_lo  = (S1 - H) / n1
    mean_H2_lo = (S2 - H*H) / n1

    C_lo = (beta**2 / N_sites) * (mean_H2_lo - mean_H_lo**2)
    e_lo = mean_H_lo / N_sites

    def jk_stderr(theta_lo):
        tbar = theta_lo.mean()
        var = (n - 1.0) / n * np.mean((theta_lo - tbar)**2)
        return np.sqrt(var)

    C_se = jk_stderr(C_lo)
    e_se = jk_stderr(e_lo)

    return (C, C_se), (e_mean, e_se)

# ---------------------------
# Single-seed run
# ---------------------------
def run_single_seed(seed):
    rng = np.random.default_rng(seed)
    x = rng.choice(STATES, size=(L, L))  # random initial configuration

    per_T = []  # list of dicts, one per temperature

    for T in temps:
        beta = 1.0 / T

        # Equilibration
        for _ in range(N_eq):
            x, h = heat_bath_parallel(x, beta, J, W_fft, rng)

        # Measurements (store time series for jackknife)
        m_series = np.empty(N_meas, dtype=np.float64)
        H_series = np.empty(N_meas, dtype=np.float64)

        for t in range(N_meas):
            x, h = heat_bath_parallel(x, beta, J, W_fft, rng)
            m_series[t] = x.mean()
            H_series[t] = energy_total(x, h)  # total energy

        # Within-seed estimates + jackknife SEs
        (mabs_mean, mabs_se), (chi, chi_se), (U4, U4_se) = jackknife_stats_m_series(m_series, beta)
        (C, C_se), (e_mean, e_se) = jackknife_stats_energy_series(H_series, beta)

        per_T.append({
            "T": float(T),
            "m_abs": (float(mabs_mean), float(mabs_se)),
            "chi":   (float(chi), float(chi_se)),
            "U4":    (float(U4), float(U4_se)),
            "C":     (float(C), float(C_se)),
            "E_per_spin": (float(e_mean), float(e_se)),
            "seed": seed
        })

    return per_T

# ---------------------------
# Aggregate across seeds
# ---------------------------
def combine_seeds(seed_results):
    """
    seed_results: list of lists (len = n_seeds) where each inner list has per-T dicts.
    Returns list of per-T dicts with combined mean and a single combined error:
       err_total = sqrt( (mean within-seed SE)^2 + (SE of seed means)^2 )
    """
    n_seeds = len(seed_results)
    n_T = len(seed_results[0])
    out = []

    for t_idx in range(n_T):
        T = seed_results[0][t_idx]["T"]

        row = {"T": T}
        for key in ["m_abs", "chi", "U4", "C", "E_per_spin"]:
            means = np.array([seed_results[s][t_idx][key][0] for s in range(n_seeds)], dtype=np.float64)
            ses   = np.array([seed_results[s][t_idx][key][1] for s in range(n_seeds)], dtype=np.float64)

            mean_val = means.mean()
            mean_internal_se = ses.mean()
            # standard error of the seed means (between-seed)
            if n_seeds > 1:
                se_across = means.std(ddof=1) / np.sqrt(n_seeds)
            else:
                se_across = 0.0

            # single reported error: quadrature combination
            err_total = float(np.sqrt(mean_internal_se**2 + se_across**2))

            row[key] = (float(mean_val), err_total)

        out.append(row)

    return out

# ---------------------------
# Report helpers
# ---------------------------
def estimate_Tc_from_chi(combined):
    chis = np.array([c["chi"][0] for c in combined])
    Ts   = np.array([c["T"] for c in combined])
    idx = int(np.argmax(chis))
    return float(Ts[idx])

def print_table(combined, seeds_used):
    print(f"Seeds: {seeds_used}")
    Tc_est = estimate_Tc_from_chi(combined)
    print(f"Estimated transition temperature (from combined susceptibility peak): T* ≈ {Tc_est:.4f}")
    print("T,  <|m|>±err,  chi±err,  U4±err,  C±err,  E/N±err")
    for r in combined:
        T = r["T"]
        m_abs, m_abs_e = r["m_abs"]
        chi, chi_e = r["chi"]
        U4, U4_e = r["U4"]
        C, C_e = r["C"]
        e, e_e = r["E_per_spin"]
        print(f"{T:7.3f}, {m_abs:.6f}±{m_abs_e:.6f}, {chi:.6f}±{chi_e:.6f}, "
              f"{U4:.6f}±{U4_e:.6f}, {C:.6f}±{C_e:.6f}, {e:.6f}±{e_e:.6f}")

# ---------------------------
# Main
# ---------------------------
if __name__ == "__main__":
    # Run all seeds
    all_seed_results = []
    for sd in seeds:
        res = run_single_seed(sd)
        all_seed_results.append(res)

    # Combine across seeds into a single mean ± error per T/observable
    combined = combine_seeds(all_seed_results)

    # Print results
    print_table(combined, seeds_used=seeds)


Seeds: [123456, 789011, 424242]
Estimated transition temperature (from combined susceptibility peak): T* ≈ 0.8247
T,  <|m|>±err,  chi±err,  U4±err,  C±err,  E/N±err
  5.000, 0.012083±0.000199, 0.187520±0.004612, 0.013336±0.035041, 0.315579±0.019059, 0.684669±0.000617
  4.898, 0.011610±0.000229, 0.180409±0.006418, -0.068732±0.020789, 0.289450±0.009629, 0.651828±0.001550
  4.796, 0.011347±0.000343, 0.169103±0.011358, 0.053589±0.055426, 0.317511±0.014784, 0.621555±0.000988
  4.694, 0.011582±0.000557, 0.182546±0.015860, 0.021831±0.057881, 0.281324±0.012061, 0.590634±0.000797
  4.593, 0.011309±0.000318, 0.179729±0.008896, -0.042843±0.021492, 0.287234±0.002094, 0.560709±0.000823
  4.491, 0.011849±0.000189, 0.199867±0.004336, 0.007869±0.041618, 0.309623±0.035134, 0.529616±0.001319
  4.389, 0.011713±0.000307, 0.199637±0.007452, -0.005131±0.062536, 0.301137±0.014090, 0.497962±0.001732
  4.287, 0.011025±0.000177, 0.186016±0.004768, -0.056755±0.077711, 0.313205±0.009818, 0.468533±0.000811
  4.185

In [5]:

import math
import random
from collections import defaultdict

def ln_double_factorial_odd(n):
    # For odd double factorial (n must be odd): n!! = (n+1)! / (2^((n+1)/2) * ((n+1)/2)!)
    # Here for N=50, |Ω| = 49!! = 50! / (2^25 * 25!)
    # Return ln(49!!)
    # More general version: ln((2k-1)!!) = ln((2k)!) - k ln 2 - ln(k!)
    # For N=50 => k=25
    k = 25
    return math.lgamma(2*k + 1) - k*math.log(2.0) - math.lgamma(k + 1)

def unit_circle_points(N):
    pts = []
    for j in range(N):
        theta = 2.0 * math.pi * j / N
        pts.append((math.cos(theta), math.sin(theta)))
    return pts

def random_matching(N, rng):
    # Generate a uniform random perfect matching via random shuffle and pair up
    idx = list(range(N))
    rng.shuffle(idx)
    pairs = [(idx[2*i], idx[2*i+1]) for i in range(N//2)]
    return pairs

def segment_intersection(p, r, q, s, eps=1e-12):
    # Intersection of segments p->p+r and q->q+s (r=p2-p, s=q2-q)
    # Returns (intersects, t, u, point)
    rx, ry = r
    sx, sy = s
    rxs = rx*sy - ry*sx
    qmpx = q[0] - p[0]
    qmpy = q[1] - p[1]
    qmpxr = qmpx*ry - qmpy*rx
    qmpxs = qmpx*sy - qmpy*sx
    if abs(rxs) < 1e-16:
        # Parallel or colinear; ignore as non-generic for random matchings
        return (False, None, None, None)
    t = qmpxs / rxs
    u = qmpxr / rxs
    # Strict interior intersection only (exclude endpoints) to avoid duplicating endpoints
    if t > eps and t < 1.0 - eps and u > eps and u < 1.0 - eps:
        ix = p[0] + t*rx
        iy = p[1] + t*ry
        return (True, t, u, (ix, iy))
    return (False, None, None, None)

def round_key(pt, tol=1e-12):
    return (round(pt[0]/tol)*tol, round(pt[1]/tol)*tol)

def build_planar_graph(N, pairs, pts, eps=1e-12):
    # Build vertices: start with endpoints (0..N-1)
    # Add interior intersections as new vertices
    # For each chord, split by its interior vertices and add edges between consecutive points
    # Output:
    # - vertices: list of (x,y)
    # - edges: list of (u,v) undirected
    # - half_edges adjacency for face traversal
    # - a map is_endpoint[v] to filter boundary-touching faces
    # Compute all intersections
    chords = []
    for a,b in pairs:
        p = pts[a]; q = pts[b]
        chords.append((a, b, p, q))

    # Map exact (endpoint) vertices
    vertices = list(pts)
    is_endpoint = [True]*N
    vid_map = {round_key(pts[i]): i for i in range(N)}

    # For each chord, collect its split points (t, vertex id)
    chord_splits = []
    for idx, (a,b,p,q) in enumerate(chords):
        splits = [(0.0, a), (1.0, b)]
        r = (q[0]-p[0], q[1]-p[1])
        chord_splits.append(splits)

    # Collect interior intersections, deduplicate by spatial key
    for i in range(len(chords)):
        a1,b1,p1,q1 = chords[i]
        r1 = (q1[0]-p1[0], q1[1]-p1[1])
        for j in range(i+1, len(chords)):
            a2,b2,p2,q2 = chords[j]
            r2 = (q2[0]-p2[0], q2[1]-p2[1])
            inter, t, u, ip = segment_intersection(p1, r1, p2, r2)
            if inter:
                key = round_key(ip, tol=1e-12)
                if key in vid_map:
                    vid = vid_map[key]
                else:
                    vid = len(vertices)
                    vertices.append(ip)
                    vid_map[key] = vid
                    is_endpoint.append(False)
                chord_splits[i].append((t, vid))
                chord_splits[j].append((u, vid))

    # Build edges by sorting split points along each chord
    # Deduplicate edges (u,v) with u<v
    edge_set = set()
    edges = []
    for idx, (a,b,p,q) in enumerate(chords):
        splits = chord_splits[idx]
        splits.sort(key=lambda x: x[0])
        # Convert to unique sequence of vertex ids (remove duplicates by t proximity)
        seq = []
        last_vid = None
        for t, vid in splits:
            if last_vid is None or vid != last_vid:
                seq.append(vid)
                last_vid = vid
        for s in range(len(seq)-1):
            u = seq[s]; v = seq[s+1]
            if u == v:
                continue
            e = (u,v) if u < v else (v,u)
            if e not in edge_set:
                edge_set.add(e)
                edges.append((u,v))

    # Build half-edge adjacency with angular ordering at each vertex
    # For each undirected edge (u,v), add two half-edges u->v and v->u
    out_edges = defaultdict(list)  # v -> list of (to, angle, edge_id)
    dir_edges = []  # list of (u,v)
    for (u,v) in edges:
        dir_edges.append((u,v))
        dir_edges.append((v,u))

    def angle(u, v):
        ux, uy = vertices[u]
        vx, vy = vertices[v]
        return math.atan2(vy-uy, vx-ux)

    for i, (u,v) in enumerate(dir_edges):
        ang = angle(u,v)
        out_edges[u].append((v, ang, i))

    # Sort outgoing edges at each vertex by angle increasing (CCW order)
    for u in out_edges:
        out_edges[u].sort(key=lambda x: x[1])

    # Build a helper: for a given incoming half-edge (u->v), at vertex v choose the next half-edge
    # which is the outgoing from v whose angle is the next CCW from angle(v->u)
    def next_half_edge(u, v):
        ang_prev = angle(v, u)  # direction we came from
        lst = out_edges[v]
        # binary search for first angle > ang_prev
        lo, hi = 0, len(lst)
        while lo < hi:
            mid = (lo + hi) // 2
            if lst[mid][1] <= ang_prev:
                lo = mid + 1
            else:
                hi = mid
        idx = lo % len(lst)
        return (v, lst[idx][0]), lst[idx][2]  # returns (next half-edge), index in dir_edges

    return vertices, edges, dir_edges, out_edges, is_endpoint

def walk_faces(vertices, dir_edges, out_edges, is_endpoint, area_tol=1e-12):
    V = vertices
    H = len(dir_edges)
    visited = [False]*H
    faces = []

    def poly_area(poly):
        # Shoelace area (absolute)
        a = 0.0
        n = len(poly)
        for i in range(n):
            x1,y1 = V[poly[i]]
            x2,y2 = V[poly[(i+1)%n]]
            a += x1*y2 - x2*y1
        return abs(a)*0.5

    # Build an index mapping for quick selection of next half-edge
    # Prepare a reverse map from (u->v) to half-edge index for marking visited along traversal
    he_index = {}
    for i, (u,v) in enumerate(dir_edges):
        he_index[(u,v)] = i

    for i in range(H):
        if visited[i]:
            continue
        # Start walking the face to the left of this half-edge
        start_u, start_v = dir_edges[i]
        u, v = start_u, start_v
        poly = []
        hed = i
        while True:
            visited[hed] = True
            poly.append(u)
            (nu, nv), hed = next_half_edge(u, v)  # move keeping face on left
            u, v = nu, nv
            if u == start_u and v == start_v:
                break

        # Filter: ignore faces that include any circle endpoint; and small area slivers
        # The polygon we recorded is the vertex list around the face
        if any(is_endpoint[vid] for vid in poly):
            continue
        A = poly_area(poly)
        if A < area_tol:
            continue
        k = len(poly)
        sgn = 1.0 if (k % 2 == 0) else -1.0
        faces.append((A, k, sgn))

    return faces

def energy_of_configuration(N, pairs, pts):
    vertices, edges, dir_edges, out_edges, is_endpoint = build_planar_graph(N, pairs, pts)
    faces = walk_faces(vertices, dir_edges, out_edges, is_endpoint, area_tol=1e-12)
    E = 0.0
    for A, k, sgn in faces:
        E += sgn * math.exp(A)
    return E

def estimate_logZ_once(N=50, beta=0.1, M=1000, seed=0):
    rng = random.Random(seed)
    pts = unit_circle_points(N)
    # Collect log-weights using log-sum-exp for numerical stability
    logw = []
    for m in range(M):
        pairs = random_matching(N, rng)
        E = energy_of_configuration(N, pairs, pts)
        logw.append(-beta * E)
    # log-mean-exp
    a = max(logw)
    mean_exp = sum(math.exp(x - a) for x in logw) / len(logw)
    log_mean_exp = a + math.log(mean_exp)
    ln_Omega = ln_double_factorial_odd(N-1)  # equals ln((N-1)!!)
    return ln_Omega + log_mean_exp

def run_three_estimates(N=50, beta=0.1, M=1000, seeds=(0,1,2)):
    logs = [estimate_logZ_once(N, beta, M, s) for s in seeds]
    mean_logZ = sum(logs)/len(logs)
    return logs, mean_logZ, round(mean_logZ)

if __name__ == "__main__":
    N = 50
    beta = 0.1
    M = 1000
    seeds = (0, 1, 2)
    logs, mean_logZ, rounded = run_three_estimates(N, beta, M, seeds)
    print("Individual ln Z estimates:", logs)
    print("Mean ln Z:", mean_logZ)
    print("Rounded to nearest integer:", rounded)

NameError: name 'next_half_edge' is not defined

In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import math
import random
from collections import defaultdict

# ============================================================
# Geometry primitives
# ============================================================

EPS = 1e-10
TAU = 2.0 * math.pi

def dot(a, b):
    return a[0]*b[0] + a[1]*b[1]

def cross(a, b):
    return a[0]*b[1] - a[1]*b[0]

def sub(a, b):
    return (a[0]-b[0], a[1]-b[1])

def seg_intersection(p, r, q, s):
    """
    Solve p + t r = q + u s for t,u in (0,1).
    Returns (ok, point, t, u) with strict interior test (excluding endpoints).
    """
    rxs = cross(r, s)
    qp = sub(q, p)
    if abs(rxs) < EPS:
        return (False, None, None, None)  # parallel or collinear (measure-zero for random chords)
    t = cross(qp, s) / rxs
    u = cross(qp, r) / rxs
    if t > EPS and t < 1.0 - EPS and u > EPS and u < 1.0 - EPS:
        ip = (p[0] + t*r[0], p[1] + t*r[1])
        return (True, ip, t, u)
    return (False, None, None, None)

def polygon_area(coords, poly_ids):
    """Shoelace area for a polygon listed as vertex ids in order (no repeated last)."""
    n = len(poly_ids)
    if n < 3:
        return 0.0
    s = 0.0
    for i in range(n):
        x1, y1 = coords[poly_ids[i]]
        x2, y2 = coords[poly_ids[(i+1) % n]]
        s += x1*y2 - x2*y1
    return abs(s) * 0.5

# ============================================================
# Build faces from chords via half-edge walk
# ============================================================

def build_faces_energy_from_pairs(N, pairs, circle_points, beta=1.0):
    """
    Given N points on the unit circle and a perfect matching 'pairs' (list of (i,j)),
    1) build all chord-chord intersections,
    2) build the planar graph (vertices: endpoints+crossings; edges: chord pieces),
    3) enumerate *bounded internal polygonal* faces (no boundary-arc faces),
    4) compute energy E = sum_{faces} sign * exp(area), sign=+ if even corners else -.
    Returns (E, faces_list) where faces_list = [(corners, area), ...].
    """
    # 0) Vertex list: first N are the circle points
    coords = list(circle_points)  # id: 0..N-1 are boundary points
    next_vid = N

    # 1) Gather intersections for each chord
    chords = []
    for (a, b) in pairs:
        pa, pb = coords[a], coords[b]
        chords.append((a, b, pa, pb))

    cuts = [ [] for _ in range(len(chords)) ]  # per-chord internal split points (t, vid)
    inter_vid = {}

    for i in range(len(chords)):
        a1, b1, p1, p2 = chords[i]
        r = sub(p2, p1)
        for j in range(i+1, len(chords)):
            a2, b2, q1, q2 = chords[j]
            s = sub(q2, q1)
            ok, ip, t, u = seg_intersection(p1, r, q1, s)
            if ok:
                vid = next_vid
                next_vid += 1
                coords.append(ip)
                inter_vid[(i, j)] = vid
                cuts[i].append((t, vid))
                cuts[j].append((u, vid))

    # 2) Build subsegment edges along each chord
    adj = defaultdict(set)
    directed_edges = set()

    for idx, (a, b, pa, pb) in enumerate(chords):
        pieces = [(0.0, a)] + sorted(cuts[idx], key=lambda x: x[0]) + [(1.0, b)]
        for k in range(len(pieces)-1):
            u_id = pieces[k][1]
            v_id = pieces[k+1][1]
            if u_id == v_id:
                continue
            adj[u_id].add(v_id)
            adj[v_id].add(u_id)
            directed_edges.add((u_id, v_id))
            directed_edges.add((v_id, u_id))

    # 3) Angular order of neighbors per vertex
    neighbor_order = {}
    for v, nbrs in adj.items():
        cx, cy = coords[v]
        angs = []
        for w in nbrs:
            wx, wy = coords[w]
            ang = math.atan2(wy - cy, wx - cx)
            angs.append((ang, w))
        angs.sort()
        neighbor_order[v] = [w for (ang, w) in angs]

    def next_edge(u, v):
        """
        Given (u->v), pick (v->w) with smallest positive turn relative to incoming direction.
        Walks faces CCW (interior on the left).
        """
        vx, vy = coords[v]
        ang_in = math.atan2(coords[u][1] - vy, coords[u][0] - vx)
        best_w = None
        best_delta = None
        for w in neighbor_order[v]:
            wx, wy = coords[w]
            ang_out = math.atan2(wy - vy, wx - vx)
            delta = (ang_out - ang_in) % TAU
            if delta <= EPS:
                delta += TAU
            if best_delta is None or delta < best_delta:
                best_delta = delta
                best_w = w
        return best_w

    # 4) Face-walk over half-edges
    visited = set()
    faces = []

    for (u0, v0) in list(directed_edges):
        if (u0, v0) in visited:
            continue
        poly = []
        u, v = u0, v0
        for _ in range(10_000):
            poly.append(u)
            w = next_edge(u, v)
            if w is None:
                poly = []
                break
            visited.add((u, v))
            u, v = v, w
            if (u, v) == (u0, v0):
                break
        else:
            poly = []

        if not poly:
            continue
        if poly[0] == poly[-1]:
            poly = poly[:-1]
        if len(poly) < 3:
            continue
        if any(vid < N for vid in poly):  # discard faces touching boundary
            continue

        A = polygon_area(coords, poly)
        if A <= 1e-12:
            continue

        faces.append((poly, A))

    # 5) Energy: ±exp(area); + for even corners, − for odd
    E = 0.0
    for poly, A in faces:
        k = len(poly)
        sign = +1.0 if (k % 2 == 0) else -1.0
        E += sign * math.exp(A)

    return E, [(len(poly), A) for (poly, A) in faces]

# ============================================================
# Sampling & thermodynamics
# ============================================================

def double_factorial_odd(n_minus_1):
    """(N-1)!! for even N (n_minus_1 = N-1)."""
    n = n_minus_1
    if n < 1:
        return 1
    prod = 1
    while n > 0:
        prod *= n
        n -= 2
    return prod

def random_perfect_matching(N):
    """Uniform random pairing of 0..N-1."""
    idx = list(range(N))
    random.shuffle(idx)
    pairs = [(idx[2*i], idx[2*i+1]) for i in range(N // 2)]
    return pairs

def circle_points(N, radius=1.0):
    """Equally spaced points on a unit circle (counterclockwise)."""
    pts = []
    for k in range(N):
        theta = TAU * (k / N)
        pts.append((radius * math.cos(theta), radius * math.sin(theta)))
    return pts

def logsumexp(values):
    """Return log(sum_i exp(values_i)) stably."""
    if not values:
        return -math.inf
    m = max(values)
    if not math.isfinite(m):
        return m
    s = sum(math.exp(v - m) for v in values)
    return m + math.log(s)

def estimate_Z_and_Eavg(N, beta=1.0, samples=400, rng_seed=None):
    """
    Monte Carlo estimate of:
      Z(beta) ≈ (N-1)!! * mean_{uniform pairings} [exp(-beta E)]
      <E> ≈ sum E exp(-beta E) / sum exp(-beta E)
    Returns dict including 'logZ_est' and 'E_avg'.
    """
    if N % 2 == 1 or N < 2:
        raise ValueError("N must be an even integer ≥ 2.")

    if rng_seed is not None:
        random.seed(rng_seed)

    pts = circle_points(N)
    log_weights = []
    energies = []

    for _ in range(samples):
        pairs = random_perfect_matching(N)
        E, _faces = build_faces_energy_from_pairs(N, pairs, pts, beta=beta)
        energies.append(E)
        log_weights.append(-beta * E)

    # <E> using normalized log-weights
    L = logsumexp(log_weights)
    E_avg = sum(Ei * math.exp(li - L) for Ei, li in zip(energies, log_weights))

    # log Z estimate
    matchings = double_factorial_odd(N - 1)  # (N-1)!!
    log_mean_w = logsumexp(log_weights) - math.log(samples)
    log_Z_est = math.log(matchings) + log_mean_w

    return {
        "E_avg": E_avg,
        "logZ_est": log_Z_est,
        "samples": samples,
        "beta": beta,
        "matchings": matchings,
    }

# ============================================================
# CLI (fixed N=100; average over 3 independent runs)
# ============================================================

def mean_std(xs):
    n = len(xs)
    mu = sum(xs) / n
    if n < 2:
        return mu, 0.0
    var = sum((x - mu) ** 2 for x in xs) / (n - 1)
    return mu, math.sqrt(var)

def main():
    # Fixed problem size
    N = 50
    BETA = 0.1
    SAMPLES = 400  # per-run samples (sane default for N=100)
    RUNS = 3
    SEEDS = [12, 122, 2222]  # independent runs; tweak if you like

    logZ_vals = []
    Eavg_vals = []

    for r in range(RUNS):
        seed = SEEDS[r % len(SEEDS)]
        out = estimate_Z_and_Eavg(N, beta=BETA, samples=SAMPLES, rng_seed=seed)
        logZ_vals.append(out["logZ_est"])
        Eavg_vals.append(out["E_avg"])

    mu_logZ, sd_logZ = mean_std(logZ_vals)
    mu_E, sd_E = mean_std(Eavg_vals)

    print("=== Random chords on a circle (N=100) ===")
    print(f"β = {BETA}, samples/run = {SAMPLES}, runs = {RUNS}")
    print("Per-run results:")
    for i, (lz, e) in enumerate(zip(logZ_vals, Eavg_vals), 1):
        print(f"  Run {i}: log Z = {lz:.10f},  <E> = {e:.10f}")
    print("\nAverages over runs:")
    print(f"  log Z  mean ± std: {mu_logZ:.10f} ± {sd_logZ:.10f}")
    print(f"  <E>    mean ± std: {mu_E:.10f} ± {sd_E:.10f}")

if __name__ == "__main__":
    main()


=== Random chords on a circle (N=100) ===
β = 0.1, samples/run = 400, runs = 3
Per-run results:
  Run 1: log Z = 75.2085210341,  <E> = -25.5178804451
  Run 2: log Z = 75.3099946638,  <E> = -28.2365973376
  Run 3: log Z = 75.2085918923,  <E> = -24.9600048007

Averages over runs:
  log Z  mean ± std: 75.2423691967 ± 0.0585653832
  <E>    mean ± std: -26.2381608611 ± 1.7530309671


In [13]:
import numpy as np
from scipy.spatial import ConvexHull
from collections import defaultdict
import itertools

def generate_circle_points(N):
    """Generate N equally spaced points on unit circle"""
    angles = np.linspace(0, 2*np.pi, N, endpoint=False)
    return np.column_stack([np.cos(angles), np.sin(angles)])

def random_matching(N):
    """Generate random perfect matching of N points"""
    indices = np.arange(N)
    np.random.shuffle(indices)
    return [(indices[i], indices[i+1]) for i in range(0, N, 2)]

def line_intersection(p1, p2, p3, p4):
    """Find intersection of line segments p1-p2 and p3-p4"""
    x1, y1 = p1
    x2, y2 = p2
    x3, y3 = p3
    x4, y4 = p4
    
    denom = (x1-x2)*(y3-y4) - (y1-y2)*(x3-x4)
    if abs(denom) < 1e-10:
        return None
    
    t = ((x1-x3)*(y3-y4) - (y1-y3)*(x3-x4)) / denom
    u = -((x1-x2)*(y1-y3) - (y1-y2)*(x1-x3)) / denom
    
    if 0 < t < 1 and 0 < u < 1:
        x = x1 + t * (x2 - x1)
        y = y1 + t * (y2 - y1)
        return (x, y)
    return None

def find_chord_intersections(points, matching):
    """Find all intersection points between chords"""
    intersections = []
    chord_intersections = defaultdict(list)  # chord_idx -> list of (intersection_point, other_chord_idx)
    
    for i, (a1, b1) in enumerate(matching):
        for j, (a2, b2) in enumerate(matching[i+1:], i+1):
            inter = line_intersection(points[a1], points[b1], points[a2], points[b2])
            if inter:
                intersections.append(inter)
                chord_intersections[i].append((inter, j))
                chord_intersections[j].append((inter, i))
    
    return intersections, chord_intersections

def build_graph_and_find_faces(points, matching, intersections, chord_intersections):
    """Build planar graph and enumerate internal faces"""
    # Build adjacency structure for planar graph traversal
    graph = defaultdict(list)
    
    # For each chord, order intersection points along the chord
    for chord_idx, (a, b) in enumerate(matching):
        chord_points = [points[a]]
        
        # Get intersections on this chord and sort by distance from endpoint a
        inters_on_chord = chord_intersections[chord_idx]
        if inters_on_chord:
            inters_sorted = sorted(inters_on_chord, 
                                 key=lambda x: np.linalg.norm(np.array(x[0]) - points[a]))
            for inter, _ in inters_sorted:
                chord_points.append(inter)
        
        chord_points.append(points[b])
        
        # Add edges between consecutive points on the chord
        for i in range(len(chord_points)-1):
            p1 = tuple(chord_points[i])
            p2 = tuple(chord_points[i+1])
            graph[p1].append(p2)
            graph[p2].append(p1)
    
    # Find faces using planar face traversal
    faces = find_planar_faces(graph)
    
    # Filter to keep only internal faces (not touching circle boundary)
    circle_points = set(map(tuple, points))
    internal_faces = []
    
    for face in faces:
        # Check if face touches circle boundary
        if not any(p in circle_points for p in face):
            internal_faces.append(face)
    
    return internal_faces

def find_planar_faces(graph):
    """Find all faces in a planar graph using cycle detection"""
    visited_edges = set()
    faces = []
    
    for start_vertex in graph:
        for next_vertex in graph[start_vertex]:
            edge = (start_vertex, next_vertex)
            if edge in visited_edges or (next_vertex, start_vertex) in visited_edges:
                continue
            
            # Traverse face using right-hand rule
            face = trace_face(graph, start_vertex, next_vertex, visited_edges)
            if face and len(face) >= 3:
                faces.append(face)
    
    return faces

def trace_face(graph, start, current, visited_edges):
    """Trace a face in planar graph using consistent turning"""
    face = [start]
    prev = start
    curr = current
    
    while True:
        face.append(curr)
        visited_edges.add((prev, curr))
        
        if curr == start and len(face) > 2:
            return face[:-1]  # Remove duplicate start point
        
        if len(face) > 100:  # Prevent infinite loops
            return None
        
        # Find next vertex using planar embedding (rightmost turn)
        neighbors = graph[curr]
        if len(neighbors) < 2:
            return None
        
        # Get angle of incoming edge
        v_in = np.array(prev) - np.array(curr)
        angle_in = np.arctan2(v_in[1], v_in[0])
        
        # Find neighbor with smallest positive angle difference
        best_next = None
        min_angle = float('inf')
        
        for neighbor in neighbors:
            if neighbor == prev:
                continue
            
            v_out = np.array(neighbor) - np.array(curr)
            angle_out = np.arctan2(v_out[1], v_out[0])
            
            # Compute angle difference (turn right)
            angle_diff = (angle_in - angle_out) % (2 * np.pi)
            
            if angle_diff < min_angle:
                min_angle = angle_diff
                best_next = neighbor
        
        if best_next is None:
            return None
        
        prev = curr
        curr = best_next

def compute_face_area(face):
    """Compute area using shoelace formula"""
    n = len(face)
    if n < 3:
        return 0
    
    area = 0
    for i in range(n):
        j = (i + 1) % n
        area += face[i][0] * face[j][1]
        area -= face[j][0] * face[i][1]
    
    return abs(area) / 2

def compute_configuration_energy(points, matching):
    """Compute total energy for a configuration"""
    # Find intersections
    intersections, chord_intersections = find_chord_intersections(points, matching)
    
    # Find internal faces
    faces = build_graph_and_find_faces(points, matching, intersections, chord_intersections)
    
    # Compute energy
    total_energy = 0
    for face in faces:
        area = compute_face_area(face)
        if area < 1e-12:  # Skip slivers
            continue
        
        k = len(face)  # Number of corners
        if k % 2 == 0:
            energy = np.exp(area)
        else:
            energy = -np.exp(area)
        
        total_energy += energy
    
    return total_energy

def estimate_log_partition_function(N, beta, M, seed=None):
    """Estimate log Z using Monte Carlo sampling"""
    if seed is not None:
        np.random.seed(seed)
    
    points = generate_circle_points(N)
    energies = []
    
    for _ in range(M):
        matching = random_matching(N)
        E = compute_configuration_energy(points, matching)
        energies.append(E)
    
    # Estimate partition function using log-sum-exp trick
    energies = np.array(energies)
    min_E = np.min(-beta * energies)
    
    log_Z = min_E + np.log(np.mean(np.exp(-beta * energies - min_E)))
    
    return log_Z

# Run 3 independent estimates
N = 50
beta = 0.1
M = 1000

results = []
for seed in [42, 123, 789]:
    log_Z = estimate_log_partition_function(N, beta, M, seed)
    results.append(log_Z)
    print(f"Run {len(results)}: log Z = {log_Z:.2f}")

mean_log_Z = np.mean(results)
print(f"\nMean log Z = {mean_log_Z:.2f}")
print(f"Rounded to nearest integer: {int(round(mean_log_Z))}")

Run 1: log Z = 1.24
Run 2: log Z = 1.29
Run 3: log Z = 1.23

Mean log Z = 1.26
Rounded to nearest integer: 1
